# DiGToR - Figures + Gap-Closing Notebook

End-to-end notebook that (1) renders **real signal images** (disagreement `d`, reliability
`c_V`/`c_T`, routing maps) to drop into the architecture figure, and (2) fills the two
rubric gaps that hold the project at ~8.4/10 - the **Ablation Study** and the **Error
Analysis** - to push toward ~9.5.

**Sections**
0. Config | 1. Pull checkpoints (W&B) | 2. Load models | 3. Architecture signal images |
4. Qualitative routing (Fig 4) | 5. Ablation study | 6. Deep metrics + loss curves |
7. Error analysis (Fig 13)

Self-contained on Colab Pro / Kaggle (clone -> data -> checkpoints from W&B -> render),
mirroring `digtor-fmb.ipynb`. Edit the **Config** cell, then Run-All.

## 0a. Clone the repo (fresh = latest main, with the module-ablation flags)

In [ ]:
import os
REPO_NAME = "DiGToR"
REPO_URL  = "https://github.com/nguyenmaiductrong/DiGToR.git"
if os.path.isdir("digtor"):
    print("already inside the repo:", os.getcwd())
else:
    if not os.path.isdir(REPO_NAME):
        print("cloning", REPO_URL); os.system(f"git clone {REPO_URL}")
    os.chdir(REPO_NAME)
# always pull latest main so train.py/models.py have the --ablate_* module flags
os.system("git pull origin main")
print("cwd:", os.getcwd())

## 0. Config - edit these

In [ ]:
import os, sys, json
from pathlib import Path

# ===== EDIT THESE =====
DATASET       = "fmb"            # "fmb" or "semanticrt"
DATA_ROOT     = "/kaggle/input/fmb/FMB"          # dataset root (has train/ test/)
WANDB_PROJECT = "digtor-fmb"     # set to None to skip W&B and use local CKPT_DIR
WANDB_ENTITY  = None             # your wandb entity/username, or None
WANDB_ALIAS   = "latest"
CKPT_DIR      = "ckpt_fmb"       # where checkpoints land / are read from
BASE          = 32
HEIGHT, WIDTH = 384, 512
SEED          = 42
IGNORE_BG     = (DATASET == "semanticrt")   # fold class-0 background into ignore
# ======================

REPO = Path.cwd()
if not (REPO / "digtor").exists():
    for cand in [REPO, REPO.parent, Path("/kaggle/working/DiGToR"),
                 Path("/content/DiGToR"), Path("/content/drive/MyDrive/DiGToR")]:
        if (cand / "digtor").exists():
            REPO = cand; break
sys.path.insert(0, str(REPO)); os.chdir(REPO)
FIG = REPO / "fig_assets"; FIG.mkdir(exist_ok=True)
RES = REPO / "results_nb"; RES.mkdir(exist_ok=True)
print("repo     :", REPO)
print("dataset  :", DATASET, "| ignore_bg:", IGNORE_BG)
print("data root:", DATA_ROOT, "->", "OK" if Path(DATA_ROOT).exists() else "MISSING")

## 0b. Auto-download dataset from Google Drive (idempotent)
Mirrors the proven gdown logic in `digtor-fmb.ipynb` / `digtor-semanticrt.ipynb`. Skips the download if `DATA_ROOT` is already populated.

In [ ]:
import glob, zipfile, shutil, subprocess

def _have(p):
    return bool(p) and Path(p).exists() and any(Path(p).rglob("*"))

if not _have(DATA_ROOT):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    base = "/content" if Path("/content").is_dir() else str(REPO.parent)
    if DATASET == "fmb":
        URL = "https://drive.google.com/drive/folders/1T_jVi80tjgyHTQDpn-TjfySyW4CK1LlF"
        DATA_ROOT = f"{base}/FMB"; mods = ("Visible", "Infrared", "Label")
        ready = lambda: all((Path(DATA_ROOT) / s / m).is_dir()
                            for s in ("train", "test") for m in mods)
        if not ready():
            dl = str(REPO / "_dl"); shutil.rmtree(dl, ignore_errors=True)
            gdown.download_folder(URL, output=dl, quiet=False, use_cookies=False)
            find = lambda n: glob.glob(f"{dl}/**/{n}", recursive=True)[0]
            def _modroot(b):
                for r, _, _ in os.walk(b):
                    if all((Path(r) / m).is_dir() for m in mods): return r
                raise FileNotFoundError("no Visible/Infrared/Label under " + b)
            for sp in ("train", "test"):
                tmp = str(REPO / f"_uz_{sp}"); shutil.rmtree(tmp, ignore_errors=True)
                with zipfile.ZipFile(find(f"{sp}.zip")) as z: z.extractall(tmp)
                dst = f"{DATA_ROOT}/{sp}"; shutil.rmtree(dst, ignore_errors=True)
                shutil.move(_modroot(tmp), dst); shutil.rmtree(tmp, ignore_errors=True)
            shutil.rmtree(dl, ignore_errors=True)
    else:
        FID = "1KXgqYLy-yQBLAzhXY1zusVQiQ27Gr0Uw"
        DATA_ROOT = f"{base}/SemanticRT_dataset"; mods = ("rgb", "thermal", "labels")
        ready = lambda: (all((Path(DATA_ROOT) / m).is_dir() for m in mods)
                         and (Path(DATA_ROOT) / "test.txt").is_file())
        if not ready():
            z = str(REPO / "_semrt.zip"); gdown.download(id=FID, output=z, quiet=False)
            tmp = str(REPO / "_uz"); shutil.rmtree(tmp, ignore_errors=True)
            with zipfile.ZipFile(z) as zz: zz.extractall(tmp)
            def _droot(b):
                for r, _, _ in os.walk(b):
                    if all((Path(r) / m).is_dir() for m in mods): return r
                raise FileNotFoundError("no rgb/thermal/labels under " + b)
            shutil.rmtree(DATA_ROOT, ignore_errors=True)
            shutil.move(_droot(tmp), DATA_ROOT)
            shutil.rmtree(tmp, ignore_errors=True); os.remove(z)
print("DATA_ROOT =", DATA_ROOT, "->", "OK" if _have(DATA_ROOT) else "MISSING")

## 1. Pull checkpoints from W&B
Needs `wandb login` or `WANDB_API_KEY` in the environment. Skips gracefully if absent.

In [ ]:
if WANDB_PROJECT:
    try:
        import wandb  # noqa: F401
    except ImportError:
        os.system("pip -q install wandb")
    from digtor.wandb_ckpt import pull_checkpoint
    got = []
    for m in ["digtor", "v_only", "t_only", "fusion"]:
        if pull_checkpoint(m, CKPT_DIR, WANDB_PROJECT, WANDB_ENTITY, WANDB_ALIAS):
            got.append(m)
    # Extra seeds for the 3-seed 5A ablation. Artifact digtor-seed{N}-ckpt lands
    # at CKPT_DIR/digtor-seed{N}.pt (train them with notebooks/digtor-fmb-seeds.ipynb).
    # Missing seeds are skipped, so 5A still runs (as 1 seed) before they exist.
    for m in ["digtor-seed0", "digtor-seed1"]:
        if pull_checkpoint(m, CKPT_DIR, WANDB_PROJECT, WANDB_ENTITY, WANDB_ALIAS):
            got.append(m)
    print("pulled:", got)
else:
    print("WANDB_PROJECT=None -> expecting local checkpoints in", CKPT_DIR)

for m in ["digtor", "v_only", "t_only", "fusion", "digtor-seed0", "digtor-seed1"]:
    print(f"  {m}.pt:", (Path(CKPT_DIR) / f"{m}.pt").exists())

## 2. Load models + test loader

In [ ]:
import numpy as np, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd

from digtor import (IGNORE_INDEX, get_dataset_config, get_dataset_module, enable_fast_gpu)
from digtor.models import DiGToR, UNet, TwoStreamFusion
from digtor.metrics import (confusion_matrix, metrics_from_cm, correctness_mask,
                            four_region_partition, aggregate_rescue,
                            corrupt_thermal, corrupt_visible, corruption_seed)

enable_fast_gpu()
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = get_dataset_config(DATASET); ds = get_dataset_module(DATASET)
NC  = cfg.num_classes
ign_classes = (0,) if IGNORE_BG else ()

split = ds.deterministic_split(DATA_ROOT, seed=SEED)
_, _, test_loader = ds.build_loaders(DATA_ROOT, size=(HEIGHT, WIDTH),
                                     batch_size=1, num_workers=2, split=split, seed=SEED)
print("test images:", len(split["test"]), "| classes:", NC, "| device:", device)

def _load(model, path):
    sd = torch.load(path, map_location=device)["model"]
    model.load_state_dict(sd, strict=False)   # strict=False: tolerate older ckpts
    return model.to(device).eval()

digtor = _load(DiGToR(base=BASE, num_classes=NC), f"{CKPT_DIR}/digtor.pt")
mv     = _load(UNet(in_ch=3, base=BASE, num_classes=NC), f"{CKPT_DIR}/v_only.pt")
mt     = _load(UNet(in_ch=1, base=BASE, num_classes=NC), f"{CKPT_DIR}/t_only.pt")
print("learned reliability-gate lambda:", round(float(digtor.rel_gate_lambda), 4))

## 3. Architecture signal images (Fig 2)

Renders the **real** per-pixel signals the model actually computes - disagreement `d`,
reliability `c_V`, `c_T`, the 3-colour routing map, and the 4-region GT map - to replace
the image-gen placeholders in `kientruc.png`. Saved to `fig_assets/arch_*.png`.

In [ ]:
# branch colours (V-trust blue, T-rescue red, Joint green) and region colours
PATH_RGB   = np.array([[0x42,0x87,0xf5],[0xf0,0x5a,0x3c],[0x78,0xc8,0x5a]], np.uint8)
REGION_RGB = np.array([[0xf0,0x5a,0x3c],[0x42,0x87,0xf5],[0xb0,0xb0,0xb0],[0xf5,0xc0,0x42]], np.uint8)

def to_img(t):
    x = t.detach().cpu().numpy()
    if x.shape[0] == 1: x = np.repeat(x, 3, 0)
    return np.clip(x.transpose(1, 2, 0), 0, 1)

def colorize_labels(lbl, nc):
    cmap = (plt.get_cmap("tab20")(np.arange(nc) % 20)[:, :3] * 255).astype(np.uint8)
    out = np.zeros((*lbl.shape, 3), np.uint8)
    m = (lbl >= 0) & (lbl < nc)
    out[m] = cmap[lbl[m]]
    return out

def _up(mapt, size):
    return F.interpolate(mapt, size=size, mode="bilinear", align_corners=False)

@torch.no_grad()
def signals_for(rgb, ir):
    logit, aux = digtor(rgb, ir, hard=True, return_aux=True)
    H, W = rgb.shape[-2:]
    return dict(
        d  = _up(aux["d"],  (H, W))[0, 0].cpu().numpy(),     # bottleneck, upsampled
        cv = _up(aux["cv"], (H, W))[0, 0].cpu().numpy(),
        ct = _up(aux["ct"], (H, W))[0, 0].cpu().numpy(),
        pi = aux["pi"][0].argmax(0).cpu().numpy(),            # full-res routing 0/1/2
        pred = logit.argmax(1)[0].cpu().numpy(),
        cv_fine = aux["cv_s"][0][0, 0].cpu().numpy(),         # full-res reliability
        ct_fine = aux["ct_s"][0][0, 0].cpu().numpy(),
    )

@torch.no_grad()
def regions_for(rgb, ir, gt):
    pv = mv(rgb).argmax(1)[0].cpu().numpy()
    pt = mt(ir).argmax(1)[0].cpu().numpy()
    g = gt.copy()
    if IGNORE_BG: g[g == 0] = IGNORE_INDEX
    parts = four_region_partition(pv, pt, g, IGNORE_INDEX)
    reg = np.full(g.shape, 2, np.int32)
    reg[parts["v_preserve"]] = 1; reg[parts["t_rescue"]] = 0; reg[parts["hard"]] = 3
    return reg, pv, pt

# Pick the most compelling RESCUE scene: RGB-only fails, thermal is right, and
# DiGToR routes those pixels to the Thermal-Rescue path AND predicts them correctly.
# Set SCENE_INDEX to a test index to pin a specific frame instead.
SCENE_INDEX = None
SCAN_N = 400
best = None
for idx, batch in enumerate(test_loader):
    if idx >= SCAN_N: break
    rgb = batch["rgb"].to(device); ir = batch["ir"].to(device); gt = batch["label"].numpy()[0]
    reg, pv, pt = regions_for(rgb, ir, gt)
    t_resc = (reg == 0)
    if SCENE_INDEX is None and t_resc.mean() < 0.02:
        continue
    s = signals_for(rgb, ir)
    af = correctness_mask(s["pred"], gt)
    rescued = t_resc & af & (s["pi"] == 1)        # rescued correctly via the thermal path
    score = int(rescued.sum())
    if SCENE_INDEX == idx:
        best = (score, rgb, ir, gt, reg, s, idx); break
    if SCENE_INDEX is None and (best is None or score > best[0]):
        best = (score, rgb, ir, gt, reg, s, idx)
assert best is not None, "no thermal-rescue scene found in the scanned range"
score, rgb, ir, gt, reg, s, sidx = best
rescued_pct = 100 * score / max((reg == 0).sum(), 1)
print(f"chosen rescue scene: test idx {sidx} | {score} px ({rescued_pct:.1f}% of t_rescue) "
      f"rescued correctly via the thermal path")
ir_gray = ir[0, 0].cpu().numpy()
ir_norm = (ir_gray - ir_gray.min()) / (np.ptp(ir_gray) + 1e-6)   # min-max normalised thermal

plt.imsave(FIG / "arch_rgb.png", to_img(rgb[0]))
plt.imsave(FIG / "arch_thermal.png", ir_norm, cmap="inferno")            # normalised thermal
plt.imsave(FIG / "arch_disagreement.png", s["d"], cmap="RdBu_r")
plt.imsave(FIG / "arch_reliability_cv.png", s["cv"], cmap="Blues", vmin=0, vmax=1)
plt.imsave(FIG / "arch_reliability_ct.png", s["ct"], cmap="Oranges", vmin=0, vmax=1)
plt.imsave(FIG / "arch_routing.png", PATH_RGB[s["pi"]])                  # routing map
plt.imsave(FIG / "arch_segmentation.png", colorize_labels(s["pred"], NC))  # model output seg map
plt.imsave(FIG / "arch_gt.png", colorize_labels(gt, NC))                 # GT seg map
plt.imsave(FIG / "arch_region.png",  REGION_RGB[reg])
print("saved 9 arch assets ->", FIG)
for p in ["rgb", "thermal", "disagreement", "reliability_cv", "reliability_ct",
          "routing", "segmentation", "gt", "region"]:
    print("  arch_%s.png" % p, "OK" if (FIG / f"arch_{p}.png").exists() else "MISSING")

panels = [(to_img(rgb[0]), "RGB", None), (ir_norm, "Thermal (norm)", "inferno"),
          (s["d"], "Disagreement d", "RdBu_r"), (s["cv"], "c_V", "Blues"),
          (s["ct"], "c_T", "Oranges"), (PATH_RGB[s["pi"]], "Routing", None),
          (colorize_labels(s["pred"], NC), "Segmentation", None)]
fig, ax = plt.subplots(1, 7, figsize=(24, 3.2))
for a, (im, ti, cm) in zip(ax, panels):
    a.imshow(im, cmap=cm); a.set_title(ti); a.axis("off")
plt.tight_layout(); plt.show()

## 4. Qualitative routing across scenes (Fig 4)

`RGB | IR | GT | DiGToR routing | GT 4-region` for several rescue scenes - the figure
reviewers find most convincing. Saved to `fig_assets/fig4_qualitative.png`.

In [ ]:
rows, MAXR = [], 4
for batch in test_loader:
    if len(rows) >= MAXR: break
    rgb = batch["rgb"].to(device); ir = batch["ir"].to(device); gt = batch["label"].numpy()[0]
    reg, pv, pt = regions_for(rgb, ir, gt)
    if (reg == 0).mean() < 0.02: continue
    s = signals_for(rgb, ir)
    rows.append((to_img(rgb[0]), ir[0, 0].cpu().numpy(), colorize_labels(gt, NC),
                 PATH_RGB[s["pi"]], REGION_RGB[reg]))

cols = ["RGB", "IR", "GT", "Routing (V/T/Joint)", "GT regions"]
fig, ax = plt.subplots(len(rows), 5, figsize=(18, 3.4 * len(rows)))
ax = np.atleast_2d(ax)
for r, row in enumerate(rows):
    for c, im in enumerate(row):
        cm = "inferno" if c == 1 else None
        ax[r, c].imshow(im, cmap=cm); ax[r, c].axis("off")
        if r == 0: ax[r, c].set_title(cols[c])
plt.tight_layout(); plt.savefig(FIG / "fig4_qualitative.png", dpi=200, bbox_inches="tight")
plt.show()
print("legend: V-trust=blue, T-rescue=red, Joint=green | regions: t_rescue=red, v_preserve=blue, easy=grey, hard=yellow")

## 5. Ablation Study

Two tiers:
- **5A - inference-time ablations (free, from the single `digtor.pt`).** The reliability
  gate and soft/hard routing are *inference* switches, so we can toggle them with no
  retraining. This already gives a real, defensible ablation table.
- **5B - module ablations (retrain).** Removing an architectural module (router signal
  set / coarse->fine routing / projection heads / Joint path / reliability gate) needs a
  retrain. Run-All trains the 5 ablated models on FMB (teachers reused from W&B) and
  tabulates each module's contribution.

### 5A. Inference-time ablations (no retraining)

In [ ]:
@torch.no_grad()
def eval_variant(model, corrupt=None, rel_gate=None, hard=True, limit=None):
    """corrupt: None or (where, kind, strength). rel_gate: None=learned lambda, 0=gate off."""
    cm = np.zeros((NC, NC), np.int64); route = np.zeros(3, np.int64)
    gen = torch.Generator(device=device)
    for idx, batch in enumerate(test_loader):
        if limit and idx >= limit: break
        rgb = batch["rgb"].to(device); ir = batch["ir"].to(device); gt = batch["label"].numpy()[0]
        gen.manual_seed(corruption_seed(SEED, idx))
        if corrupt:
            where, kind, st = corrupt
            if where == "thermal": ir = corrupt_thermal(ir, kind, st, gen=gen)
            else:                  rgb = corrupt_visible(rgb, kind, st, gen=gen)
        logit, aux = model(rgb, ir, hard=hard, return_aux=True, rel_gate=rel_gate)
        route += np.bincount(aux["pi"].argmax(1).cpu().numpy().ravel(), minlength=3)
        cm += confusion_matrix(logit.argmax(1)[0].cpu().numpy(), gt, NC, IGNORE_INDEX)
    m = metrics_from_cm(cm, ign_classes)
    m["route_share"] = (route / max(route.sum(), 1)).tolist()
    return m

configs = [
    ("DiGToR (full, learned lambda)", dict(rel_gate=None, hard=True)),
    ("- reliability gate (lambda=0)", dict(rel_gate=0.0,  hard=True)),
    ("soft routing (vs hard argmax)", dict(rel_gate=None, hard=False)),
]
# 3-seed 5A. seed 42 = canonical digtor.pt; seeds 0/1 = digtor-seed{N}.pt pulled in
# Section 1 (produce them with notebooks/digtor-fmb-seeds.ipynb). Same fixed test set
# and same corruption seed, so only the trained weights vary -> the std is pure seed
# noise. Missing seeds are skipped, so this still runs as 1 seed before they exist.
seed_ckpts = {42: f"{CKPT_DIR}/digtor.pt",
              0:  f"{CKPT_DIR}/digtor-seed0.pt",
              1:  f"{CKPT_DIR}/digtor-seed1.pt"}
metrics = ["clean_mIoU", "th_dropout_mIoU", "th_noise0.6_mIoU", "route_T_share"]
per_seed = {}
for sd, path in seed_ckpts.items():
    if not Path(path).exists():
        print("skip seed", sd, "(no ckpt)"); continue
    m = _load(DiGToR(base=BASE, num_classes=NC), path)
    rows = []
    for name, kw in configs:
        clean = eval_variant(m, None, **kw)
        drop  = eval_variant(m, ("thermal", "dropout", 1.0), **kw)
        noise = eval_variant(m, ("thermal", "noise", 0.6), **kw)
        rows.append({"variant": name, "clean_mIoU": clean["mIoU"],
                     "th_dropout_mIoU": drop["mIoU"], "th_noise0.6_mIoU": noise["mIoU"],
                     "route_T_share": clean["route_share"][1]})
    per_seed[sd] = pd.DataFrame(rows).set_index("variant")
    print("done seed", sd)

variants = per_seed[next(iter(per_seed))].index
abl_mean = pd.DataFrame({mt: pd.concat([per_seed[s][mt] for s in per_seed], axis=1).mean(axis=1)
                         for mt in metrics}, index=variants)
abl_std = pd.DataFrame({mt: pd.concat([per_seed[s][mt] for s in per_seed], axis=1).std(axis=1, ddof=1)
                        for mt in metrics}, index=variants)
if len(per_seed) > 1:
    abl_inf = abl_mean.copy()
    for mt in metrics:
        abl_inf[mt] = [f"{abl_mean.loc[v, mt]:.4f} ± {abl_std.loc[v, mt]:.4f}" for v in variants]
else:
    abl_inf = abl_mean.round(4)
abl_inf.to_csv(RES / "ablation_inference.csv")
print(f"5A over seeds {sorted(per_seed)} (n={len(per_seed)}):")
print("Read: gate ~neutral on CLEAN but should LIFT mIoU under thermal failure "
      "(dropout/noise) -> that delta IS the gate's measured contribution; the std "
      "shows whether it survives seed noise.")
abl_inf

In [ ]:
# bar chart: gate ON (full) vs OFF, clean vs thermal-dropout. Error bars = seed std.
gate_rows = [v for v in abl_mean.index if ("gate" in v or "full" in v)]
sub_m = abl_mean.loc[gate_rows]; sub_s = abl_std.loc[gate_rows]
multi = len(per_seed) > 1
x = np.arange(len(gate_rows)); w = 0.35
plt.figure(figsize=(6, 4))
plt.bar(x - w/2, sub_m["clean_mIoU"], w, label="clean",
        yerr=sub_s["clean_mIoU"].values if multi else None, capsize=4)
plt.bar(x + w/2, sub_m["th_dropout_mIoU"], w, label="thermal dropout",
        yerr=sub_s["th_dropout_mIoU"].values if multi else None, capsize=4)
plt.xticks(x, ["gate ON", "gate OFF"]); plt.ylabel("mIoU"); plt.legend()
plt.title("Reliability-gate ablation (inference-time)"
          + (f", {len(per_seed)} seeds (mean±std)" if multi else ""))
plt.tight_layout(); plt.savefig(FIG / "fig12_gate_ablation.png", dpi=200); plt.show()

### 5B. Module ablations (RETRAINS - Run-All does these)

Each variant removes exactly **one architectural module** from DiGToR and retrains, so the
mIoU/TRR drop vs `full` measures that module's contribution. **FMB only.** The two teachers
(`v_only`, `t_only`) are **loaded from W&B** (Section 1) and used as fixed KD targets - they
are **never retrained**.

| variant | module removed |
|---|---|
| `abl_signals_d_only` | reliability signals into the router (router sees disagreement only) |
| `abl_routing_bottleneck` | coarse->fine hierarchical skip routing (route at bottleneck only) |
| `abl_proj_raw` | task-aligned projection heads (disagreement from raw cosine) |
| `abl_paths_2` | the Joint-fusion path (2-path router) |
| `abl_gate_on` | adds the reliability gate back (full FMB rig trains gate-off) |

WARNING: Trains 5 models (~5x one digtor run). Idempotent (skips existing). `RUN_5B=False` to skip;
small `ABL_EPOCHS` for a smoke test.

In [ ]:
RUN_5B     = True       # set False to skip the retrains entirely
ABL_EPOCHS = 80         # match the main digtor run; e.g. 3 for a quick smoke test
ABL_BS     = 8

assert DATASET == "fmb", "5B module ablations are configured for FMB only."
for tk in ["v_only", "t_only"]:
    assert Path(f"{CKPT_DIR}/{tk}.pt").exists(), \
        f"missing teacher {CKPT_DIR}/{tk}.pt -- pull it from W&B in Section 1 first"

# MAIN digtor recipe (FMB Phase-0). Teachers are LOADED (--v_ckpt/--t_ckpt), not retrained.
MAIN_FLAGS = ("--amp --lr 5e-4 --ignore_bg --corrupt_aug --corrupt_p 0.5 "
              "--gamma_prior 2.0 --lambda_cost 0.1 --route_beta 0.7 "
              "--lambda_distill 0.5 --disable_gate")
base = (f"--dataset fmb --root {DATA_ROOT} --mode digtor --epochs {ABL_EPOCHS} "
        f"--bs {ABL_BS} --height {HEIGHT} --width {WIDTH} --base {BASE} "
        f"--v_ckpt {CKPT_DIR}/v_only.pt --t_ckpt {CKPT_DIR}/t_only.pt {MAIN_FLAGS}")

# each variant removes exactly ONE architectural module vs the full model
abl_variants = {
    "abl_signals_d_only":     base + " --ablate_signals d_only",
    "abl_routing_bottleneck": base + " --ablate_routing bottleneck_only",
    "abl_proj_raw":           base + " --ablate_proj raw",
    "abl_paths_2":            base + " --ablate_paths 2",
    "abl_gate_on":            base.replace("--disable_gate", ""),   # add the reliability gate
}
if not RUN_5B:
    print("RUN_5B=False -> skipping retrains; 5B-eval tabulates whatever exists.")
for name, flags in (abl_variants.items() if RUN_5B else []):
    out = f"ckpt_abl/{name}"
    if Path(f"{out}/digtor.pt").exists():
        print(f"[skip] {name}: {out}/digtor.pt exists"); continue
    cmd = f"python -m digtor.train {flags} --out {out}"
    print("\n>>", name, "\n  ", cmd)
    rc = os.system(cmd)
    print(f"[{name}] exit={rc}", "OK" if rc == 0 else "FAILED")

In [ ]:
# 5B-eval: tabulate every module-ablation checkpoint under ckpt_abl/<name>/digtor.pt.
# The eval applies the SAME module switch the model was trained with. Gate-off models
# eval at rel_gate=0 (FMB reporting convention); abl_gate_on uses its learned lambda.
ABL_ATTRS = {
    "full":                   {},
    "abl_signals_d_only":     dict(ablate_signals="d_only"),
    "abl_routing_bottleneck": dict(ablate_routing="bottleneck_only"),
    "abl_proj_raw":           dict(ablate_proj="raw"),
    "abl_paths_2":            dict(ablate_paths=2),
    "abl_gate_on":            {},
}
GATE_ON = {"abl_gate_on"}      # eval with learned lambda; everything else at rel_gate=0

@torch.no_grad()
def eval_ckpt(path, attrs=None, rel_gate=0.0):
    m = DiGToR(base=BASE, num_classes=NC)
    for k, v in (attrs or {}).items():
        setattr(m, k, v)
    m.load_state_dict(torch.load(path, map_location=device)["model"], strict=False)
    m = m.to(device).eval()
    cm = np.zeros((NC, NC), np.int64); route = np.zeros(3, np.int64)
    pvs, pts, pfs, gts = [], [], [], []
    for batch in test_loader:
        rgb = batch["rgb"].to(device); ir = batch["ir"].to(device); gt = batch["label"].numpy()[0]
        logit, aux = m(rgb, ir, hard=True, return_aux=True, rel_gate=rel_gate)
        pf = logit.argmax(1)[0].cpu().numpy()
        cm += confusion_matrix(pf, gt, NC, IGNORE_INDEX)
        route += np.bincount(aux["pi"].argmax(1).cpu().numpy().ravel(), minlength=3)
        g = gt.copy()
        if IGNORE_BG: g[g == 0] = IGNORE_INDEX
        pvs.append(mv(rgb).argmax(1)[0].cpu().numpy())
        pts.append(mt(ir).argmax(1)[0].cpu().numpy()); pfs.append(pf); gts.append(g)
    seg = metrics_from_cm(cm, ign_classes)
    resc = aggregate_rescue(pvs, pts, pfs, gts, IGNORE_INDEX)
    return {"mIoU": round(seg["mIoU"], 4), "FWIoU": round(seg["FWIoU"], 4),
            "TRR": round(resc["TRR"], 4), "HRR": round(resc["HRR"], 4),
            "route_T": round((route / max(route.sum(), 1))[1], 3)}

cands = [("full", f"{CKPT_DIR}/digtor.pt")]
abl_dir = Path("ckpt_abl")
if abl_dir.exists():
    cands += sorted((p.parent.name, str(p)) for p in abl_dir.glob("*/digtor.pt"))
rows = []
for n, p in cands:
    if not Path(p).exists(): continue
    rg = None if n in GATE_ON else 0.0
    rows.append({"variant": n, **eval_ckpt(p, ABL_ATTRS.get(n, {}), rg)})
abl_train = pd.DataFrame(rows)
abl_train.to_csv(RES / "ablation_module.csv", index=False)
print("Each row removes ONE module vs 'full'. mIoU/TRR drop vs full = that module's contribution.")
abl_train

## 6. Deep result metrics
Reuses the project's eval CLIs (no logic duplicated), then loads their JSON. Loss curves are already logged live to the **W&B dashboard** during training (`digtor-fmb` project) - reference/screenshot them there, no need to re-plot here.

In [ ]:
IGN = "--ignore_bg" if IGNORE_BG else ""
os.system(f'python -m digtor.eval_rescue   --dataset {DATASET} --root "{DATA_ROOT}" '
          f'--ckpt_dir {CKPT_DIR} {IGN} --out {RES}/rescue.json')
os.system(f'python -m digtor.eval_detector --dataset {DATASET} --root "{DATA_ROOT}" '
          f'--v_ckpt {CKPT_DIR}/v_only.pt --t_ckpt {CKPT_DIR}/t_only.pt {IGN} --out {RES}/detector.json')
os.system(f'python -m digtor.eval_robustness --dataset {DATASET} --root "{DATA_ROOT}" '
          f'--ckpt_dir {CKPT_DIR} {IGN} --out {RES}/robustness.json')

for name in ["rescue", "detector", "robustness"]:
    p = RES / f"{name}.json"
    if p.exists():
        print(f"\n===== {name}.json =====")
        print(json.dumps(json.loads(p.read_text()), indent=2)[:1500])

## 7. Error Analysis (Fig 13)

Finds and explains the model's most informative failures. The headline failure mode is
**thermal crossover**: the thermal reliability `c_T` stays high (the gate keeps trusting
thermal) but thermal is actually wrong, so the router sends those pixels to **Thermal-
Rescue** and the final prediction is wrong. We rank test images by this failure mass and
render annotated panels.

In [ ]:
@torch.no_grad()
def scan_failures(max_imgs=400):
    recs = []
    for idx, batch in enumerate(test_loader):
        if idx >= max_imgs: break
        rgb = batch["rgb"].to(device); ir = batch["ir"].to(device); gt = batch["label"].numpy()[0]
        s = signals_for(rgb, ir)
        g = gt.copy()
        if IGNORE_BG: g[g == 0] = IGNORE_INDEX
        valid = g != IGNORE_INDEX
        pt = mt(ir).argmax(1)[0].cpu().numpy()
        at = correctness_mask(pt, g); af = correctness_mask(s["pred"], g)
        tpick = (s["pi"] == 1) & valid                  # router chose Thermal-Rescue
        cross = tpick & (~at) & (~af) & (s["ct_fine"] > 0.5)   # but thermal wrong & c_T high
        recs.append({"idx": idx, "cross_fail": int(cross.sum()),
                     "err_rate": float((~af & valid).sum()) / max(int(valid.sum()), 1),
                     "ct_in_fail": float(s["ct_fine"][cross].mean()) if cross.sum() else 0.0})
    return sorted(recs, key=lambda r: -r["cross_fail"])

fails = scan_failures()
top_idx = [r["idx"] for r in fails[:3]]
print("Top thermal-crossover failure images (idx, #fail px, mean c_T there):")
for r in fails[:3]:
    print(f"  img {r['idx']:4d}  cross_fail={r['cross_fail']:6d}  c_T={r['ct_in_fail']:.3f}")

In [ ]:
# render annotated panels for the worst cases
def err_overlay(rgb_img, wrong):
    out = rgb_img.copy()
    out[wrong] = out[wrong] * 0.25 + np.array([1, 0, 0]) * 0.75
    return out

panels_done = 0
for idx, batch in enumerate(test_loader):
    if idx not in top_idx: continue
    rgb = batch["rgb"].to(device); ir = batch["ir"].to(device); gt = batch["label"].numpy()[0]
    s = signals_for(rgb, ir)
    g = gt.copy()
    if IGNORE_BG: g[g == 0] = IGNORE_INDEX
    valid = g != IGNORE_INDEX
    wrong = (~correctness_mask(s["pred"], g)) & valid
    rgb_img = to_img(rgb[0])
    cols = [(rgb_img, "RGB", None), (ir[0, 0].cpu().numpy(), "IR", "inferno"),
            (colorize_labels(gt, NC), "GT", None),
            (colorize_labels(s["pred"], NC), "DiGToR pred", None),
            (err_overlay(rgb_img, wrong), "Errors (red)", None),
            (PATH_RGB[s["pi"]], "Routing", None),
            (s["ct_fine"], "c_T", "Oranges")]
    fig, ax = plt.subplots(1, 7, figsize=(24, 3.4))
    for a, (im, ti, cm) in zip(ax, cols):
        a.imshow(im, cmap=cm, vmin=0 if cm == "Oranges" else None,
                 vmax=1 if cm == "Oranges" else None)
        a.set_title(ti); a.axis("off")
    plt.tight_layout(); plt.savefig(FIG / f"fig13_case{panels_done}.png", dpi=200,
                                    bbox_inches="tight"); plt.show()
    r = next(x for x in fails if x["idx"] == idx)
    print(f"CASE {panels_done} (img {idx}): {r['cross_fail']} px routed to Thermal-Rescue "
          f"with mean c_T={r['ct_in_fail']:.2f} are WRONG. Physical cause: thermal crossover "
          f"- sun-warmed background matches target temperature, so c_T stays high and the gate "
          f"keeps trusting a thermal signal that has lost contrast. Fix direction: a crossover-"
          f"aware reliability target (penalise high c_T where thermal contrast collapses).")
    panels_done += 1

In [ ]:
# quantitative error breakdown: worst classes + per-region accuracy
cm = np.zeros((NC, NC), np.int64); pvs, pts, pfs, gts = [], [], [], []
for batch in test_loader:
    rgb = batch["rgb"].to(device); ir = batch["ir"].to(device); gt = batch["label"].numpy()[0]
    pf = digtor(rgb, ir, hard=True).argmax(1)[0].cpu().numpy()
    cm += confusion_matrix(pf, gt, NC, IGNORE_INDEX)
    g = gt.copy()
    if IGNORE_BG: g[g == 0] = IGNORE_INDEX
    pvs.append(mv(rgb).argmax(1)[0].cpu().numpy())
    pts.append(mt(ir).argmax(1)[0].cpu().numpy()); pfs.append(pf); gts.append(g)

seg = metrics_from_cm(cm, ign_classes)
iou = seg["per_class_iou"]
worst = sorted([(cfg.class_names[i], v) for i, v in enumerate(iou)
                if v is not None and (i not in ign_classes)], key=lambda t: t[1])[:5]
resc = aggregate_rescue(pvs, pts, pfs, gts, IGNORE_INDEX)
print("Overall  mIoU=%.4f  FWIoU=%.4f" % (seg["mIoU"], seg["FWIoU"]))
print("\n5 worst classes by IoU (where errors concentrate):")
for n, v in worst: print(f"  {n:14s} IoU={v:.3f}")
print("\nPer-region accuracy (Thermal Rescue Protocol):")
for k in ["TRR", "VPR", "HRR", "EasyAcc"]:
    print(f"  {k:8s} = {resc[k]:.4f}")

---
### Outputs
- `fig_assets/arch_{rgb,thermal,disagreement,reliability_cv,reliability_ct,routing,segmentation,gt,region}.png`
  - the 9 real maps to drop into the **architecture figure** (thermal is min-max normalised)
- `fig_assets/fig4_qualitative.png` - qualitative routing (Fig 4)
- `fig_assets/fig12_gate_ablation.png` + `results_nb/ablation_*.csv` - **Ablation study**
- `fig_assets/fig13_case*.png` - **Error analysis** with physical causes
- `results_nb/{rescue,detector,robustness}.json` - deep metrics (loss curves live on W&B)

Section 5B and the worst-case panels are the two rubric gaps; running 5B's retrains
completes the full ablation table.